# CNN heat-model training (Colab)

Trains the PyTorch CNN heat model (S5/C2) -- predicts `lst_bicubic10` from
Sentinel-2 + spectral-index + land-cover patches -- on a GPU. Run this from
**VS Code**: `Select Kernel` -> `Colab` -> sign in -> `New Colab Server` ->
GPU (free T4 is enough) -> connect.

**Prerequisite**: this notebook needs the land-cover ensemble raster, which
can only be produced locally (U-Net inference + RF combined via
`scripts/build_landcover_ensemble.py`) -- Colab can't regenerate it. Run
`train_unet.ipynb` and the local U-Net inference steps first, then push the
resulting ensemble raster once (and after every relabel/retrain):

```bash
python -c "from src.utils import gcs; from config.settings import GCS_MODEL_BUCKET, ENSEMBLE_RASTER_GCS_PREFIX; gcs.upload_file('data/processed/landcover/ensemble_landcover.tif', GCS_MODEL_BUCKET, f'{ENSEMBLE_RASTER_GCS_PREFIX}.tif')"
```

**One-time setup**: none needed in advance -- same as `train_unet.ipynb`, the
auth cell below prompts a masked paste-in box (`getpass`) for your
service-account JSON key each session.

In [13]:
# --- Repo sync (same pattern as train_unet.ipynb) --------------------------
import os
import subprocess
import sys

REPO_URL = "https://github.com/EngineerKX/urban-heat-cooling-priority.git"
REPO_DIR = "/content/urban-heat-cooling-priority"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Repo ready at", REPO_DIR)

Repo ready at /content/urban-heat-cooling-priority


In [14]:
%pip install -q -r {REPO_DIR}/requirements-colab.txt


In [15]:
# --- Auth: service account, not interactive Google login -------------------
# Paste the full contents of your service-account JSON key when prompted
# below (open credentials/nus-iss-urban-heat-sg-*.json in a text editor,
# copy everything, Ctrl+V into the box that appears). Uses getpass -- a
# plain stdin prompt, standard Jupyter protocol -- not a browser widget,
# since google.colab.files.upload() hung indefinitely through VS Code's
# remote-kernel connection to Colab (confirmed 2026-08-05). getpass also
# never echoes the input back into the cell's saved output, unlike a plain
# input() -- this repo is public on GitHub, so that distinction matters.
import getpass
import os

key_json = getpass.getpass("Paste the full contents of your service-account JSON key, then press Enter: ")

key_path = "/content/gee_key.json"
with open(key_path, "w") as f:
    f.write(key_json)

os.environ["GEE_PRIVATE_KEY_PATH"] = key_path
os.environ["GEE_SERVICE_ACCOUNT"] = "urban-heat-pipeline@nus-iss-urban-heat-sg.iam.gserviceaccount.com"
os.environ["GEE_PROJECT_ID"] = "nus-iss-urban-heat-sg"
os.environ["GEE_EXPORT_BUCKET"] = "nus-iss-urban-heat-sg-exports"
print("Service-account credentials staged.")

Service-account credentials staged.


In [16]:
import json
import time
from pathlib import Path

import ee
import mlflow
import torch

from config import settings
from config.settings import (
    CNN_MODEL_SAVE_PATH,
    DRY_SEASON_MONTHS,
    ENSEMBLE_RASTER_GCS_PREFIX,
    GCS_MODEL_BUCKET,
    GEE_EXPORT_BUCKET,
    LANDSAT_CLOUD_COVER_MAX,
    NATIVE_SCALE_M,
    S2_CLOUD_PROB_MAX,
    S2_UTM_CRS,
    SG_BBOX,
    TARGET_SCALE_M,
    YEARS,
)
from src.downscaling.variants import build_lst_30m, variant_bicubic10
from src.heat_model.cnn_data import build_local_feature_target_patches
from src.heat_model.cnn_train import train_cnn_regressor
from src.ingest.gee import export_geotiff_to_gcs, init_ee
from src.ingest.subzones import as_ee_feature_collection, dissolve_boundary, fetch_subzones_geojson
from src.landcover.rf_baseline import build_feature_image
from src.landcover.unet_data import INFERENCE_PATCH_DIR, export_inference_patches
from src.utils import gcs
from src.utils.experiment_tracking import HEAT_MODEL_EXPERIMENT_NAME, export_run_summary, start_run
from src.utils.seed import set_all_seeds

In [17]:
set_all_seeds()
init_ee()
print("GPU available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

GPU available: True - Tesla T4


In [18]:
# --- Pull the ensemble raster (the one input Colab can't self-generate) ---
ENSEMBLE_RASTER_PATH = Path("data/processed/landcover/ensemble_landcover.tif")
ENSEMBLE_RASTER_PATH.parent.mkdir(parents=True, exist_ok=True)
gcs.download_blob(GCS_MODEL_BUCKET, f"{ENSEMBLE_RASTER_GCS_PREFIX}.tif", ENSEMBLE_RASTER_PATH)
print(f"Ensemble raster ready at {ENSEMBLE_RASTER_PATH}")

Ensemble raster ready at data/processed/landcover/ensemble_landcover.tif


In [19]:
# --- Build the same GEE feature image U-Net used (no labels needed here) --
sg_bbox = ee.Geometry.Rectangle(list(SG_BBOX))
subzones_fc = as_ee_feature_collection(fetch_subzones_geojson())
boundary = dissolve_boundary(subzones_fc)
feature_image, _valid_mask = build_feature_image(sg_bbox, boundary, YEARS, DRY_SEASON_MONTHS, S2_CLOUD_PROB_MAX)
print("Feature image built.")

Dissolved Singapore boundary built. Approx area: 788.3 km²
(Sanity check: Singapore's land area is ~730-735 km².)
Feature image built.


In [20]:
# --- Export (or reuse the GCS-cached) inference patches --------------------
# Same patches U-Net's local inference uses -- existence-cached in GCS, so
# this is a fast download if train_unet.ipynb or a local inference run
# already produced them, and a real (one-time) GEE export otherwise.
inference_patch_dir = export_inference_patches(feature_image, boundary)
mixer_json_path = inference_patch_dir / "unet_inference.json"

Inference patches already downloaded at /content/urban-heat-cooling-priority/data/interim/unet_patches/inference — skipping export (pass force=True to redo).


In [21]:
# --- Pull (or export) lst_bicubic10 -----------------------------------------
# The only new GEE computation S5's CNN half needs -- variant_bicubic10
# reused unmodified over the same season window as the production heat
# variants. Already sitting in GCS from earlier work, so this is normally
# just a download; the export branch exists for a from-scratch setup.
LST_BICUBIC10_GCS_PREFIX = "heat_model/lst_bicubic10_full"
LST_BICUBIC10_PATH = Path("data/interim/lst_bicubic10_full.tif")
LST_BICUBIC10_PATH.parent.mkdir(parents=True, exist_ok=True)

if gcs.blob_exists(GEE_EXPORT_BUCKET, f"{LST_BICUBIC10_GCS_PREFIX}.tif"):
    print("lst_bicubic10 already exported — downloading from GCS instead of re-running the GEE export.")
    gcs.download_blob(GEE_EXPORT_BUCKET, f"{LST_BICUBIC10_GCS_PREFIX}.tif", LST_BICUBIC10_PATH)
else:
    lst_30m = build_lst_30m(sg_bbox, YEARS, DRY_SEASON_MONTHS, LANDSAT_CLOUD_COVER_MAX, S2_UTM_CRS, NATIVE_SCALE_M)
    bicubic_image = variant_bicubic10(lst_30m, S2_UTM_CRS, TARGET_SCALE_M)
    export_geotiff_to_gcs(
        bicubic_image, description="lst_bicubic10_full", bucket=GEE_EXPORT_BUCKET,
        prefix=LST_BICUBIC10_GCS_PREFIX, region=sg_bbox, scale=TARGET_SCALE_M, crs=S2_UTM_CRS,
        out_path=LST_BICUBIC10_PATH,
    )

lst_bicubic10 already exported — downloading from GCS instead of re-running the GEE export.


In [22]:
# --- Build (X, y, valid_mask) patches, channels-first (n, C, H, W) --------
X, y, valid_mask = build_local_feature_target_patches(
    inference_patch_dir, mixer_json_path, ENSEMBLE_RASTER_PATH, LST_BICUBIC10_PATH,
)

Loaded 1066 S2/index feature patches (mixer total: 1066).
  ...built 200/1066 patches
  ...built 400/1066 patches
  ...built 600/1066 patches
  ...built 800/1066 patches
  ...built 1000/1066 patches
Valid pixels: 40.9% of 17,465,344 total.


In [23]:
# --- Train -------------------------------------------------------------
with start_run("cnn", experiment_name=HEAT_MODEL_EXPERIMENT_NAME):
    mlflow.log_params({
        "n_patches": int(X.shape[0]),
        "n_channels": int(X.shape[1]),
        "patch_size": int(X.shape[2]),
        "cnn_base_filters": settings.UNET_BASE_FILTERS,
        "cnn_epochs": settings.UNET_EPOCHS,
        "cnn_learning_rate": settings.UNET_LEARNING_RATE,
        "cnn_batch_size": settings.UNET_BATCH_SIZE,
        "cnn_early_stop_patience": settings.UNET_EARLY_STOP_PATIENCE,
    })

    model, history = train_cnn_regressor(X, y, valid_mask, force_retrain=False)

    if history is not None:
        for epoch, (loss, val_loss, val_metric) in enumerate(
            zip(history["train_loss"], history["val_loss"], history["val_metric"])
        ):
            mlflow.log_metrics({"train_loss": loss, "val_loss": val_loss, "val_rmse": val_metric}, step=epoch)
        mlflow.log_metric("best_val_loss", min(history["val_loss"]))

print(f"Model saved to {CNN_MODEL_SAVE_PATH}")

Loading cached model from /content/urban-heat-cooling-priority/models/heat_cnn.pt (pass force_retrain=True to retrain).
Model saved to /content/urban-heat-cooling-priority/models/heat_cnn.pt


In [24]:
# --- Push the trained weights + this run's MLflow summary to GCS ----------
if history is not None:
    summary = export_run_summary(
        "cnn", HEAT_MODEL_EXPERIMENT_NAME,
        params={
            "n_patches": int(X.shape[0]),
            "n_channels": int(X.shape[1]),
            "cnn_epochs": settings.UNET_EPOCHS,
            "cnn_learning_rate": settings.UNET_LEARNING_RATE,
        },
        metrics_history={
            "train_loss": history["train_loss"],
            "val_loss": history["val_loss"],
            "val_rmse": history["val_metric"],
        },
    )
    gcs.upload_text(json.dumps(summary), GCS_MODEL_BUCKET, f"training_runs/cnn_{int(time.time())}.json")
    print("Run summary pushed for local MLflow import.")

!python {REPO_DIR}/scripts/push_models.py --model cnn

[cnn] Remote copy already matches this hash — skipping push (pass --force to redo anyway).


## Next steps (on your own machine, no GPU needed)

```bash
python scripts/pull_models.py --model cnn
```

Downloads the trained weights (verified via sha256) and imports this run into
your local MLflow store. From here, `scripts/diagnose_heat_model.py` and
`scripts/run_counterfactual.py` can run the CNN locally on CPU, and the
Streamlit app's Counterfactual Greening page runs it live.